# 通过时间反向传播


## 环境配置


## 练习8.7.1

**题目：** 假设我们拥有一个对称矩阵 $\mathbf{M} \in \mathbb{R}^{n \times n}$，其特征值为 $\lambda_i$，对应的特征向量是 $\mathbf{v}_i$（$i = 1, \dots, n$）。假设特征值的序列顺序为 $|\lambda_i| \ge |\lambda_{i+1}|$。

**1a. 证明 $\mathbf{M}^k$ 拥有特征值 $\lambda_i^k$。**

**解答：** 设 $\mathbf{M} \mathbf{v}_i = \lambda_i \mathbf{v}_i$。使用数学归纳法证明：

- 当 $k=1$ 时：$\mathbf{M}^1 \mathbf{v}_i = \lambda_i \mathbf{v}_i = \lambda_i^1 \mathbf{v}_i$，成立。
- 归纳假设：假设 $\mathbf{M}^{k-1} \mathbf{v}_i = \lambda_i^{k-1} \mathbf{v}_i$ 成立。
- 归纳步骤：$\mathbf{M}^k \mathbf{v}_i = \mathbf{M} \cdot \mathbf{M}^{k-1} \mathbf{v}_i = \mathbf{M} \cdot \lambda_i^{k-1} \mathbf{v}_i = \lambda_i^{k-1} \mathbf{M} \mathbf{v}_i = \lambda_i^{k-1} \cdot \lambda_i \mathbf{v}_i = \lambda_i^k \mathbf{v}_i$。得证。

**1b. 证明对于一个随机向量 $\mathbf{x} \in \mathbb{R}^n$，$\mathbf{M}^k \mathbf{x}$ 将有较高概率与 $\mathbf{M}$ 的特征向量 $\mathbf{v}_1$ 在一条直线上。**

**解答：** 由于 $\mathbf{M}$ 是对称矩阵，其特征向量 $\{\mathbf{v}_1, \dots, \mathbf{v}_n\}$ 构成 $\mathbb{R}^n$ 的一组正交基，因此 $\mathbf{x}$ 可唯一表示为：

$$\mathbf{x} = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \dots + c_n \mathbf{v}_n$$

对于随机 $\mathbf{x}$，通常所有 $c_i \ne 0$（概率为 1）。将 $\mathbf{M}^k$ 作用于 $\mathbf{x}$：

$$\mathbf{M}^k \mathbf{x} = c_1 \lambda_1^k \mathbf{v}_1 + c_2 \lambda_2^k \mathbf{v}_2 + \dots + c_n \lambda_n^k \mathbf{v}_n$$

提取 $\lambda_1^k$：

$$\mathbf{M}^k \mathbf{x} = \lambda_1^k \left[ c_1 \mathbf{v}_1 + c_2 \left( \frac{\lambda_2}{\lambda_1} \right)^k \mathbf{v}_2 + \dots + c_n \left( \frac{\lambda_n}{\lambda_1} \right)^k \mathbf{v}_n \right]$$

由于 $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|$（非退化情况），对于 $i \ge 2$：$\lim_{k \to \infty} (\lambda_i / \lambda_1)^k = 0$。因此当 $k$ 足够大时：$\mathbf{M}^k \mathbf{x} \approx \lambda_1^k c_1 \mathbf{v}_1 \propto \mathbf{v}_1$，即 $\mathbf{M}^k \mathbf{x}$ 收敛到主特征向量 $\mathbf{v}_1$ 的方向。这就是幂迭代法（Power Iteration）的原理。

**1c. 上述结果对于循环神经网络中的梯度意味着什么？**

**解答：** 在 BPTT 中，从时间步 $T$ 到 $t$ 的梯度传递需经过 Jacobian 矩阵连乘：

$$\frac{\partial L}{\partial \mathbf{h}_t} = \frac{\partial L}{\partial \mathbf{h}_T} \cdot \prod_{k=t}^{T-1} \mathbf{J}_k$$

其中 $\mathbf{J}_k = \partial \mathbf{h}_{k+1} / \partial \mathbf{h}_k$。平稳假设下 $\mathbf{J}_k \approx \mathbf{W}$，梯度近似为 $\mathbf{W}^{T-t}$ 的连乘。根据 1a 和 1b：

**(1) 梯度爆炸与消失：** $|\lambda_1(\mathbf{W})| > 1$ → 梯度指数级爆炸；$|\lambda_1(\mathbf{W})| < 1$ → 梯度指数级消失。

**(2) 梯度方向退化：** 当 $T-t$ 较大时，$\mathbf{W}^{T-t}$ 在任意输入向量的作用下，输出方向收敛到 $\mathbf{W}$ 的主特征向量 $\mathbf{v}_1$ 方向。这导致不同输入序列经长程 BPTT 后产生的梯度方向高度相似，不同时间步的参数更新高度相关，学习效率低下。



## 练习8.7.2

**题目：** 除了梯度截断，还有其他方法来应对循环神经网络中的梯度爆炸吗？

**解答：**

1. **权重正则化** — L2 正则化限制权重矩阵 Frobenius 范数；谱正则化直接限制最大奇异值（谱范数 $\le 1$），从源头控制特征值的模。
2. **激活函数选择** — ReLU 正半轴导数 = 1 不放大梯度（但负半轴可能导致消失）；tanh 饱和区加剧消失；LeakyReLU/ELU 在负半轴也有小梯度可缓解消失。
3. **更好的架构设计** — LSTM 通过输入门、遗忘门、输出门控制信息流，Cell State 提供梯度传播"高速公路"；GRU 为 LSTM 简化版；残差连接提供梯度直接传播路径；Layer Normalization 稳定训练过程。
4. **更好的参数初始化** — Xavier(Glorot) 初始化使各层方差一致；正交初始化保持权重矩阵特征值模 $\approx 1$，是 RNN 非常推荐的初始化策略。
5. **训练技巧** — 截断 BPTT（只回传固定步数梯度）；自适应优化器 (Adam/RMSProp) 动态调整学习率；学习率衰减随训练进程逐步降低学习率。
6. **NPU 场景** — PyPTO 通过 `pypto.RunMode` 支持 NPU 原生精度控制；fp16 + fp32 混合训练配合 loss scaling 处理动态范围限制，有效应对梯度爆炸。



---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)

